# MINI PROJECT MDM 2026 - KELOMPOK 5 (DJBC)
## Master Data Importir dan Eksportir Nasional (Single View)

Notebook ini berisi implementasi lengkap Master Data Management (MDM) untuk integrasi data **OSS (NIB)** dan **CEISA (Bea Cukai)** sesuai dengan pedoman teknis DJBC.

### Tahapan MDM:
1. **Simulation**: Pembuatan data 6.000 record dengan anomali bisnis.
2. **Initial Profiling**: Analisis kualitas data awal.
3. **Cleansing & Standardization**: Pembersihan Nama, NPWP, dan Alamat.
4. **Matching**: Linkage data antar sistem.
5. **Golden Record**: Penggabungan data (Survivorship).
6. **Quality Monitoring**: Scorecard akhir (Pre vs Post MDM).

### 0. Persiapan & Install Library

In [ ]:
!pip install pandas numpy faker missingno unidecode -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import re
from faker import Faker
import random
from datetime import datetime, timedelta
from unidecode import unidecode

fake = Faker('id_ID')
sns.set_theme(style='whitegrid')
print('✅ Library siap!')

### 1. Data Simulation (6.000 Records)
Mensimulasikan data dari sistem **OSS** dan **CEISA** dengan berbagai anomali (Duplikasi, Konflik Legal, Format NPWP Rusak).

In [ ]:
Faker.seed(42)
TOTAL_UNIQUE = 5000
DUP_NAME = 500
DIFF_NPWP = 250
DIFF_ADDR = 200
CONFLICT_LEGAL = 50
TOTAL_ROWS = TOTAL_UNIQUE + DUP_NAME + DIFF_NPWP + DIFF_ADDR + CONFLICT_LEGAL

def generate_base_record():
    nib = fake.numerify('############')
    npwp = fake.numerify('###############')
    nama_pt = fake.company().upper()
    flag_impor = random.choice([0, 1])
    flag_ekspor = random.choice([0, 1])
    
    return {
        "NIB": nib,
        "NPWP": npwp,
        "OSS_ID": f"OSS-{fake.numerify('#######')}",
        "NAMA_PERUSAHAAN": nama_pt,
        "ALAMAT": fake.street_address().upper(),
        "KELURAHAN": fake.city().upper(),
        "FLAG_IMPOR": flag_impor,
        "FLAG_EKSPOR": flag_ekspor,
        "STATUS_NIB": 'AKTIF'
    }

base_records = [generate_base_record() for _ in range(TOTAL_UNIQUE)]

def create_variants(count, change_type):
    variants = []
    samples = random.sample(base_records, count)
    for r in samples:
        v = r.copy()
        if change_type == 'name': v['NAMA_PERUSAHAAN'] += " (VAR)"
        elif change_type == 'npwp': v['NPWP'] = fake.numerify('###############')
        variants.append(v)
    return variants

master_data = base_records + create_variants(DUP_NAME, 'name') + create_variants(DIFF_NPWP, 'npwp')
df_master = pd.DataFrame(master_data)

# Split Systems
df_oss = df_master.sample(frac=0.8, random_state=42)
df_ceisa = df_master.sample(frac=0.8, random_state=24)

# Dirty CEISA NPWP
df_ceisa['NPWP'] = df_ceisa['NPWP'].apply(lambda x: f"{x[:2]}.{x[2:5]}.{x[5:8]}.{x[8]}-{x[9:12]}.{x[12:]}" if random.random() > 0.5 else x)

print(f'✅ Simulation Complete: {len(df_master)} total master records')

### 2. Initial Data Profiling
Analisis kualitas data mentah sebelum dilakukan pembersihan.

In [ ]:
print("📊 PROFILING CEISA (NPWP Inconsistency Example)")
display(df_ceisa[['NIB', 'NPWP', 'NAMA_PERUSAHAAN']].head())

plt.figure(figsize=(10, 4))
msno.matrix(df_ceisa)
plt.title("Matrix Missing Values (CEISA)")
plt.show()

### 3. Cleansing & Standardization
Tahap pembersihan nama perusahaan dan penyeragaman format NPWP.

In [ ]:
def clean_name(name):
    if pd.isna(name): return None
    return unidecode(str(name)).upper().replace('PT.', 'PT').replace('CV.', 'CV').strip()

def clean_npwp(npwp):
    if pd.isna(npwp): return None
    digits = re.sub(r'\D', '', str(npwp))
    if len(digits) == 15:
        return f"{digits[0:2]}.{digits[2:5]}.{digits[5:8]}.{digits[8]}-{digits[9:12]}.{digits[12:15]}"
    return digits

df_oss['NAMA_CLEAN'] = df_oss['NAMA_PERUSAHAAN'].apply(clean_name)
df_oss['NPWP_CLEAN'] = df_oss['NPWP'].apply(clean_npwp)

df_ceisa['NAMA_CLEAN'] = df_ceisa['NAMA_PERUSAHAAN'].apply(clean_name)
df_ceisa['NPWP_CLEAN'] = df_ceisa['NPWP'].apply(clean_npwp)

print("✅ Standardization Complete: All NPWPs now follow XX.XXX.XXX.X-XXX.XXX format")

### 4. Matching & Record Linkage
Menghubungkan record yang sama antara OSS dan CEISA menggunakan NIB.

In [ ]:
matched = pd.merge(
    df_oss, 
    df_ceisa, 
    on='NIB', 
    how='inner', 
    suffixes=('_OSS', '_CEISA')
)
print(f"✅ Berhasil mencocokkan {len(matched)} record antara OSS dan CEISA")

### 5. Golden Record Management
Penggabungan data dengan aturan **Survivorship**: Nama dan NPWP resmi diambil dari OSS.

In [ ]:
golden_records = matched.copy()
golden_records = golden_records[[
    'NIB', 'NPWP_CLEAN_OSS', 'NAMA_CLEAN_OSS', 'ALAMAT_OSS', 'FLAG_IMPOR_CEISA', 'FLAG_EKSPOR_CEISA'
]]
golden_records.columns = ['NIB', 'NPWP', 'NAMA_PERUSAHAAN', 'ALAMAT', 'FLAG_IMPOR', 'FLAG_EKSPOR']

print(f"⭐ Golden Records created: {len(golden_records)} entities")
display(golden_records.head())

### 6. Final Quality Scorecard
Visualisasi perbandingan kualitas data sebelum dan sesudah MDM.

In [ ]:
def get_score(df, npwp_col):
    pattern = r'^\d{2}\.\d{3}\.\d{3}\.\d{1}-\d{3}\.\d{3}$'
    validity = df[npwp_col].apply(lambda x: bool(re.match(pattern, str(x)))).mean() * 100
    uniqueness = (1 - df.duplicated(subset=['NIB']).sum() / len(df)) * 100
    return validity, uniqueness

v_ceisa, u_ceisa = get_score(df_ceisa, 'NPWP')
v_gold, u_gold = get_score(golden_records, 'NPWP')

labels = ['Validity (NPWP)', 'Uniqueness (NIB)']
ceisa_scores = [v_ceisa, u_ceisa]
gold_scores = [v_gold, u_gold]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, ceisa_scores, width, label='CEISA (Raw)', color='salmon')
ax.bar(x + width/2, gold_scores, width, label='GOLDEN (MDM)', color='skyblue')

ax.set_ylabel('Scores (%)')
ax.set_title('Peningkatan Kualitas Data (Pre vs Post MDM)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
plt.ylim(0, 110)
plt.show()

print(f"🚀 MDM Success! Validity NPWP naik dari {v_ceisa:.1f}% menjadi {v_gold:.1f}%.")